In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import random

scaler = StandardScaler()

pd.set_option("display.max_rows", None)     # 행 제한 해제
pd.set_option("display.max_columns", None)  # 열 제한 해제
pd.set_option("display.width", None)        # 한 줄에 다 나오게
pd.set_option("display.max_colwidth", None) # 컬럼 길이 제한 해제

DATA_PATH = "./preprocess/03_processed_data/data_set.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

In [11]:
# 분석에 안 쓰는 컬럼 제외
drop_cols = ["년도", "월", "지역"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 숫자형 변수만 선택
num_df = df.select_dtypes(include=[np.number]).dropna(how="any")

num_df = pd.DataFrame(
    scaler.fit_transform(num_df),
    columns=num_df.columns,
    index=num_df.index
)
# 피어슨 상관계수 행렬
corr = num_df.corr(method="pearson")

In [12]:
# 행렬을 긴 포맷으로 변환
corr_long = (
    corr.reset_index()
        .melt(id_vars="index", var_name="변수2", value_name="상관계수")
        .rename(columns={"index": "변수1"})
)

# 자기 자신과의 상관계수 제거
corr_long = corr_long[corr_long["변수1"] != corr_long["변수2"]]

# 변수1, 변수2 쌍 정렬해서 중복 제거
corr_long[["변수1", "변수2"]] = np.sort(corr_long[["변수1", "변수2"]], axis=1)
corr_long = corr_long.drop_duplicates(subset=["변수1", "변수2"])

# 상관계수 기준 정렬
corr_long = (
    corr_long[corr_long["상관계수"].abs() >= 0.8]  
    .sort_values(by=["변수1", "상관계수"], ascending=[True, False])  # 변수1 → 상관계수 순으로 정렬
    .reset_index(drop=True)
)

corr_long


,변수1,변수2,상관계수
0,1회충전주행거리,차량전체_등록수,0.955616
1,1회충전주행거리,연료소비효율,0.925752
2,1회충전주행거리,하이브리드차_등록수,0.897079
3,1회충전주행거리,하이브리드차_순증,0.876350
4,1회충전주행거리,수소차_등록수,0.868247
5,1회충전주행거리,차량전체_순증가량,0.808240
6,1회충전주행거리,보조금,-0.886799
7,cpi,평균기준금리,0.942273
8,급속충전기_누적대수,전체충전기_누적대수,0.947121
9,급속충전기_누적대수,완속충전기_누적대수,0.936288


In [ ]:

# 3) 종속변수 / 독립변수 분리
y = num_df["전기차 등록수"]   
X = num_df.drop(columns=["전기차 등록수"])     

# # # 다중공선성 심한 변수 제거
# X.drop(columns=[
#     "완속충전기_신규설치", 
#     "급속충전기_신규설치", 
#     "전체충전기_신규설치", 
#     "완속충전기_누적대수", 
#     "급속충전기_누적대수",
#     "평균최저기온(°C)",
#     "평균최고기온(°C)",
#     "차량전체_등록수"
#     ], 
#     inplace=True)

#VIF 계산
X_const = add_constant(X)
vif_df = pd.DataFrame()
vif_df["변수"] = X_const.columns
vif_df["VIF"] = [variance_inflation_factor(X_const.values, i) 
                 for i in range(X_const.shape[1])]

#상수항 제거
vif_df = vif_df[vif_df["변수"] != "const"].reset_index(drop=True)
vif_df


,변수,VIF
0,전기차 순증가량,1.892074
1,연료소비효율,40.934803
2,1회충전주행거리,41.767914
3,배터리충전용량,5.634285
4,전체충전기_누적대수,4.033391
5,차량전체_순증가량,8.682302
6,수소차_등록수,87.995821
7,수소차_순증가량,4.183525
8,하이브리드차_등록수,132.709885
9,하이브리드차_순증,42.751320


In [14]:

# # 종속변수, 독립변수 분리
# variables = list(X.columns)

# def compute_vif(X):
#     X_const = sm.add_constant(X)
#     return [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])][1:]

# results = []

# for k in range(len(variables)-8, 14, -1): 
#     best_r2 = -np.inf
#     best_vars = None
    
#     for _ in range(500):
#         subset = random.sample(variables, k)
#         X_sub = X[subset]
        
#         # 회귀모델
#         X_const = sm.add_constant(X_sub)
#         model = sm.OLS(y, X_const).fit()
#         r2 = model.rsquared
        
#         max_vif = max(compute_vif(X_sub))
        
#         if r2 > best_r2 and max_vif < 20:
#             best_r2 = r2
#             best_vars = subset
    
#     results.append({"변수개수": k, "최고R²": best_r2, "최적조합": best_vars})

# results_df = pd.DataFrame(results)
# print(results_df)
